In [33]:
import pandas as pd
import numpy as np
import random
from itertools import combinations
import sys
import krippendorff
from pairadigm import Pairadigm, LLMClient

# Set random seeds for reproducibility
np.random.seed(42)
random.seed(42)

In [37]:
# Load the data
data = pd.read_parquet('example/data.parquet')
pairwise_df = pd.read_parquet('example/pairwise_df.parquet')
print(f"Loaded data with {len(data)} sentences")
print(f"Columns: {data.columns.tolist()}")

Loaded data with 50 sentences
Columns: ['comment_id', 'text', 'hate_speech_score', 'CGCoT_Breakdown', 'split']


In [13]:
data.head(3)

,comment_id,text,hate_speech_score,CGCoT_Breakdown,split
0,11870,Got it. Thanks for the clarifying and congrats...,-5.65,Original Text: Got it. Thanks for the clarifyi...,train
1,22840,URL helping the rohingya refugees is also anot...,-5.60,Original Text: URL helping the rohingya refuge...,eval
2,10695,"That's why that one came here, went through th...",-5.79,"Original Text: That's why that one came here, ...",train


In [12]:
pairwise_df.head(3)

,item1,item2,item1_split,item2_split,breakdown1,breakdown2,decision,justification
0,5360,14407,train,train,Original Text: Fellow trans girl here! Love an...,"Original Text: As a Pale white Algerian, I fre...",NaN,**FINAL ANSWER:** *Neither.*\n\n**JUSTIFICATIO...
1,48374,4328,train,train,Original Text: Straight white Male but Fuckin ...,Original Text: NTA. You should tell your Dad w...,Text2,"**FINAL ANSWER:** **""Description 2""**\n\n**JUS..."
2,32319,14352,train,train,Original Text: No bitch apply enough pressure ...,"Original Text: Beat it out of her, or throw he...",Text2,**FINAL ANSWER:** **Description 2**\n\n**JUSTI...


In [48]:
def simulate_manual_annotation(score1, score2, noise_level=0.3):
    """
    Simulate manual annotation based on ground truth values with some noise.
    
    Parameters:
    - score1, score2: hate speech scores for items 1 and 2
    - noise_level: how much noise to add to decisions (0 = perfect, 1 = random)
    
    Returns:
    - 'Text1' or 'Text2' indicating which text has higher concept value
    """
    
    # Calculate true difference
    true_diff = score1 - score2
    
    # Add noise to the decision
    # The larger the true difference, the less likely we are to make an error
    error_prob = noise_level * np.exp(-abs(true_diff) * 2)  # Exponential decay based on difference
    
    if np.random.random() < error_prob:
        # Make an error - flip the decision
        return 'Text2' if true_diff > 0 else 'Text1'
    else:
        # Make correct decision
        return 'Text1' if true_diff > 0 else 'Text2'

def create_pairwise_dataset(data, pairwise_df, num_annotators=4):
    """
    Create a pairwise comparison dataset with simulated human annotations.
    """
    
    # Add text and emotion scores for both items
    id_to_data = data.set_index('comment_id').to_dict('index')
    
    pairwise_data = []
    
    for _, row in pairwise_df.iterrows():

        item1_id = row['item1']
        item2_id = row['item2']
        
        item1_data = id_to_data[item1_id]
        item2_data = id_to_data[item2_id]

        pair_row = {
            'item1_id': item1_id,
            'item2_id': item2_id,
            'item1_text': item1_data['text'],
            'item2_text': item2_data['text'],
            'item1_score': item1_data['hate_speech_score'],
            'item2_score': item2_data['hate_speech_score']
        }

        for annotator_id in range(1, num_annotators + 1):
            # Each annotator has slightly different noise levels
            noise_level = 0.2 + (annotator_id - 1) * 0.1  # 0.2, 0.3, 0.4
            
            annotation = simulate_manual_annotation(
                item1_data['hate_speech_score'], item2_data['hate_speech_score'],
                noise_level=noise_level
            )
            
            pair_row[f'A{annotator_id}'] = annotation
        
        pairwise_data.append(pair_row)
    
    return pd.DataFrame(pairwise_data)

In [ ]:
# Set random seeds for reproducibility
np.random.seed(42)
random.seed(42)

# Generate the pairwise dataset
print("Generating pairwise comparisons with simulated human annotations...")
example_paired_data = create_pairwise_dataset(
    data, pairwise_df, 
    num_annotators=10
)

print(f"Created pairwise dataset with {len(pairwise_df)} pairs")
print(f"Columns: {example_paired_data.columns.tolist()}")

# Show sample of the data
print("\nSample of pairwise data:")
print(example_paired_data.head(3))

# Convert Text1 and Text2 to 0 and 1
annotators = [c for c in example_paired_data.columns if c.startswith("A")]

example_paired_data[annotators] = (
    example_paired_data[annotators]
    .replace({"Text1": 0, "Text2": 1})
    .astype("float") 
)

alpha = krippendorff.alpha(
    reliability_data = example_paired_data[annotators].T, 
    level_of_measurement='nominal'
)

# Create a summary
print(f"Agreement/alpha: {alpha:.3f}")
print(f"\nDataset Summary:")
print(f"- Original sentences: {len(data)}")
print(f"- Pairwise comparisons: {len(example_paired_data)}")
print(f"- Annotators per concept: {len(annotators)}")
print(f"- Total annotations: {len(example_paired_data) * len(annotators)} (pairs × annotators)")

Generating pairwise comparisons with simulated human annotations...
Created pairwise dataset with 147 pairs
Columns: ['item1_id', 'item2_id', 'item1_text', 'item2_text', 'item1_score', 'item2_score', 'A1', 'A2', 'A3', 'A4', 'A5', 'A6', 'A7', 'A8', 'A9', 'A10']

Sample of pairwise data:
   item1_id  item2_id                                         item1_text  \
0      5360     14407  Fellow trans girl here! Love and support confi...   
1     48374      4328  Straight white Male but Fuckin love this song....   
2     32319     14352  No bitch apply enough pressure to make me feel...   

                                          item2_text  item1_score  \
0  As a Pale white Algerian, I frequently play on...        -8.34   
1  NTA. You should tell your Dad why you're reall...        -3.56   
2  Beat it out of her, or throw her to the curb. ...        -2.11   

   item2_score     A1     A2     A3     A4     A5     A6     A7     A8     A9  \
0        -5.80  Text2  Text2  Text2  Text2  Text2 

In [76]:
# Save the dataset
output_file = 'simulated_paired_nnotations.csv'
example_paired_data.to_csv(output_file, index=False)
print(f"\nSaved pairwise dataset to {output_file}")


Saved pairwise dataset to simulated_paired_nnotations.csv
